# 19 — MLP Autoencoder Baseline (reconstruction only, no WJ awareness)

**Purpose:** Compare against learned intersection-min MLP (nb 16) and PCA (nb 18).  
Standard autoencoder trained with MSE reconstruction loss only — no Weighted Jaccard signal.

**Architecture:**
- Encoder: `Linear(18499→4096) → BN → ReLU → Linear(4096→1024) → BN → ReLU → Linear(1024→512) → BN → ReLU → L1-normalize`
- Decoder: `Linear(512→1024) → BN → ReLU → Linear(1024→4096) → BN → ReLU → Linear(4096→18499) → ReLU`
- Loss: MSE reconstruction (no similarity loss)
- The encoder output is forced to simplex (nonneg + L1-norm) so stage-1 can use intersection — same search metric as nb 16 for a fair comparison

**Pipeline:**
- Stage-1 search: intersection (`Σmin`) on 512-D encoder embeddings (same as nb 16)
- Stage-2: exact raw WJ ratio rerank on original 18k vectors (identical to nb 16)

**Expected outcome:** Reconstruction loss pulls bottleneck away from WJ alignment → worse recall than nb 16.

**Prereqs:** `/tmp/qt_10k.npy`, `/tmp/gt_lookup_10k.pkl` from `00_cache_data.ipynb`

In [1]:
# ── Configuration ────────────────────────────────────────────────────────────
dataset_name = "10k"
run_training = True
run_eval     = True
device_str   = "cuda:0"

QUERY_START_10K  = 8000
QUERY_START_FULL = 187019

batch_size   = 512
epochs       = 50
lr           = 1e-3
weight_decay = 1e-4

candidate_ks        = [500, 1000]
rerank_batch_size   = 16
search_corpus_chunk = 8000

ckpt_path = "/tmp/best_autoencoder_10k.pt"
out_path  = "/tmp/results_autoencoder_baseline.pkl"
seed = 42

In [2]:
import gc
import pickle
import random
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

device = torch.device(device_str if torch.cuda.is_available() else "cpu")
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

print(f"device={device} | dataset={dataset_name} | train from scratch")

device=cuda:0 | dataset=10k | train from scratch


In [3]:
# ── Model ────────────────────────────────────────────────────────────────────
class QuadtreeAutoencoder(nn.Module):
    """
    MLP autoencoder on quadtree vectors.
    Encoder outputs a simplex embedding (nonneg + L1-norm) so stage-1 intersection
    search is directly comparable to nb 16.
    Decoder reconstructs the original nonneg quadtree vector.
    """
    def __init__(self, in_dim, bottleneck=512):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, 4096, bias=False), nn.BatchNorm1d(4096), nn.ReLU(),
            nn.Linear(4096, 1024, bias=False), nn.BatchNorm1d(1024), nn.ReLU(),
            nn.Linear(1024, bottleneck, bias=False), nn.BatchNorm1d(bottleneck),
        )
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck, 1024, bias=False), nn.BatchNorm1d(1024), nn.ReLU(),
            nn.Linear(1024, 4096, bias=False), nn.BatchNorm1d(4096), nn.ReLU(),
            nn.Linear(4096, in_dim, bias=False),
        )

    def encode(self, x):
        z = self.encoder(x)
        z = F.relu(z)
        z = z / z.sum(dim=1, keepdim=True).clamp(min=1e-10)  # simplex
        return z

    def decode(self, z):
        out = self.decoder(z)
        return F.relu(out)  # nonneg reconstruction (qt vectors are nonneg)

    def forward(self, x):
        return self.decode(self.encode(x))


print("QuadtreeAutoencoder defined.")

QuadtreeAutoencoder defined.


In [4]:
# ── Load data ────────────────────────────────────────────────────────────────
if dataset_name == "10k":
    qt = np.load("/tmp/qt_10k.npy")
    with open("/tmp/gt_lookup_10k.pkl", "rb") as f:
        gt = pickle.load(f)
    query_start = QUERY_START_10K
elif dataset_name == "full":
    qt = np.load("/tmp/qtree_vectors_full.npy")
    with open("/tmp/gt_lookup_full.pkl", "rb") as f:
        gt = pickle.load(f)
    query_start = QUERY_START_FULL
    ckpt_path = "/tmp/best_autoencoder_full.pt"
else:
    raise ValueError(dataset_name)

corpus_qt   = qt[:query_start]
query_qt    = qt[query_start:]
corpus_sums = corpus_qt.sum(axis=1)

print(f"qt={qt.shape} | corpus={corpus_qt.shape} | queries={query_qt.shape}")

qt=(10000, 18499) | corpus=(8000, 18499) | queries=(2000, 18499)


In [5]:
# ── Training (reconstruction only — no WJ loss) ───────────────────────────────
class ReconstructionDataset(Dataset):
    """Each sample is a single quadtree vector; target = same vector (autoencoder)."""
    def __init__(self, qtree_vectors):
        self.vecs = torch.tensor(qtree_vectors, dtype=torch.float32)

    def __len__(self):
        return len(self.vecs)

    def __getitem__(self, idx):
        return self.vecs[idx]


def train_autoencoder(qt, query_start, n_epochs):
    print("Training autoencoder from scratch (MSE reconstruction, no WJ loss).")
    model = QuadtreeAutoencoder(qt.shape[1], bottleneck=512).to(device)

    # Train on full dataset (both corpus and queries) — AE is unsupervised,
    # no GT needed, no leakage risk
    dataset = ReconstructionDataset(qt)
    loader  = DataLoader(
        dataset, batch_size=batch_size, shuffle=True,
        num_workers=4, pin_memory=device.type == "cuda", drop_last=True,
    )
    print(f"Samples={len(dataset):,} | Steps/epoch={len(loader)}")

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)

    best_loss = float("inf")
    for epoch in range(1, n_epochs + 1):
        model.train()
        total_loss, steps = 0.0, 0
        pbar = tqdm(loader, desc=f"Epoch {epoch:02d}/{n_epochs}", leave=False)
        for x in pbar:
            x = x.to(device, non_blocking=True)
            recon = model(x)
            loss  = F.mse_loss(recon, x)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += float(loss.detach().cpu())
            steps += 1
            pbar.set_postfix(loss=f"{float(loss):.6f}")

        avg_loss = total_loss / max(steps, 1)
        scheduler.step()
        if avg_loss < best_loss:
            best_loss = avg_loss
            torch.save(model.state_dict(), ckpt_path)

        if epoch == 1 or epoch % 5 == 0 or epoch == n_epochs:
            print(
                f"Epoch {epoch:02d}/{n_epochs} | loss={avg_loss:.6f} | "
                f"best={best_loss:.6f} | lr={scheduler.get_last_lr()[0]:.2e}"
            )

    print(f"Saved {ckpt_path} (best_loss={best_loss:.6f})")
    return model


if run_training:
    model = train_autoencoder(qt, query_start, epochs)
else:
    model = QuadtreeAutoencoder(qt.shape[1], bottleneck=512).to(device)
    model.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=True))
    print(f"Loaded {ckpt_path}")
model.eval()

Training autoencoder from scratch (MSE reconstruction, no WJ loss).
Samples=10,000 | Steps/epoch=19


Epoch 01/50 | loss=0.004430 | best=0.004430 | lr=9.99e-04


Epoch 05/50 | loss=0.000000 | best=0.000000 | lr=9.76e-04


Epoch 10/50 | loss=0.000000 | best=0.000000 | lr=9.05e-04


Epoch 15/50 | loss=0.000000 | best=0.000000 | lr=7.94e-04


...

Epoch 35/50 | loss=0.000000 | best=0.000000 | lr=2.06e-04


Epoch 40/50 | loss=0.000000 | best=0.000000 | lr=9.55e-05


Epoch 45/50 | loss=0.000000 | best=0.000000 | lr=2.45e-05


Epoch 50/50 | loss=0.000000 | best=0.000000 | lr=0.00e+00
Saved /tmp/best_autoencoder_10k.pt (best_loss=0.000000)


QuadtreeAutoencoder(
  (encoder): Sequential(
    (0): Linear(in_features=18499, out_features=4096, bias=False)
    (1): BatchNorm1d(4096, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Linear(in_features=4096, out_features=1024, bias=False)
    (4): BatchNorm1d(1024, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU()
    (6): Linear(in_features=1024, out_features=512, bias=False)
    (7): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (decoder): Sequential(
    (0): Linear(in_features=512, out_features=1024, bias=False)
    (1): BatchNorm1d(1024, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Linear(in_features=1024, out_features=4096, bias=False)
    (4): BatchNorm1d(4096, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU()
    (6): Linear(in_features=4096, out_features=18499, bias=False)
  )
)

In [6]:
# ── Eval helpers (same as nb 16) ─────────────────────────────────────────────
def generate_embeddings(model, data, *, dev, batch_size=512):
    model.eval()
    chunks = []
    with torch.no_grad():
        for start in tqdm(range(0, len(data), batch_size), desc="Embedding"):
            batch = torch.tensor(data[start:start+batch_size], dtype=torch.float32, device=dev)
            chunks.append(model.encode(batch).cpu().numpy())
    embs = np.vstack(chunks)
    print(f"simplex sums: min={embs.sum(1).min():.4f} max={embs.sum(1).max():.4f}")
    return embs


def recall_at_k(gt_lookup, nbrs, query_start_id, k):
    total, count = 0.0, 0
    for i, ids in enumerate(nbrs):
        qid    = query_start_id + i
        gt_set = set(gt_lookup.get(qid, [])[:k])
        if not gt_set:
            continue
        total += len(gt_set & set(ids[:k])) / len(gt_set)
        count += 1
    return total / count if count else 0.0


def eval_recall_dict(gt_lookup, nbrs, query_start_id, max_k):
    return {
        k: recall_at_k(gt_lookup, nbrs, query_start_id, k)
        for k in (10, 50, 100, 500)
        if k <= max_k
    }


@torch.no_grad()
def knn_intersection_gpu(query_embs, corpus_embs, k, dev, corpus_chunk=8000):
    """Top-k by intersection (Σmin) — same metric as nb 16."""
    q     = torch.from_numpy(query_embs).to(dev, dtype=torch.float32)
    c_all = torch.from_numpy(corpus_embs).to(dev, dtype=torch.float32)
    n_q, n_c = q.shape[0], c_all.shape[0]
    top_ids    = np.zeros((n_q, k), dtype=np.int64)
    top_scores = np.full((n_q, k), -1.0, dtype=np.float32)

    for qs in tqdm(range(0, n_q, 64), desc="KNN intersection"):
        qe  = min(qs + 64, n_q)
        qb  = q[qs:qe]
        best_scores = torch.full((qb.shape[0], k), -1.0, device=dev)
        best_ids    = torch.zeros((qb.shape[0], k), dtype=torch.long, device=dev)

        for cs in range(0, n_c, corpus_chunk):
            ce       = min(cs + corpus_chunk, n_c)
            cb       = c_all[cs:ce]
            scores   = torch.min(qb[:, None, :], cb[None, :, :]).sum(dim=2)
            cand_ids = torch.arange(cs, ce, device=dev).expand(qb.shape[0], -1)
            merged_scores = torch.cat([best_scores, scores], dim=1)
            merged_ids    = torch.cat([best_ids, cand_ids], dim=1)
            new_scores, order = torch.topk(
                merged_scores, k=min(k, merged_scores.shape[1]), dim=1
            )
            best_ids    = torch.gather(merged_ids, 1, order)
            best_scores = new_scores

        top_ids[qs:qe]    = best_ids.cpu().numpy()
        top_scores[qs:qe] = best_scores.cpu().numpy()

    return [(top_ids[i].tolist(), top_scores[i].tolist()) for i in range(n_q)]


def rerank_wj_gpu(query_qt, nbrs_ids, corpus_qt, corpus_sums, dev, batch_size=16):
    """Exact raw WJ ratio rerank on original 18k vectors — identical to nb 16."""
    corpus_t      = torch.from_numpy(corpus_qt).to(device=dev, dtype=torch.float32)
    corpus_sums_t = torch.from_numpy(corpus_sums).to(device=dev, dtype=torch.float32)
    reranked = []
    for start in tqdm(range(0, len(nbrs_ids), batch_size), desc="Raw WJ ratio rerank"):
        batch  = nbrs_ids[start:start+batch_size]
        groups = {}
        for offset, ids in enumerate(batch):
            ids_arr = np.asarray(ids, dtype=np.int64)
            groups.setdefault(len(ids_arr), []).append((start + offset, ids_arr))
        for cand_len, items in groups.items():
            if cand_len == 0:
                for _ in items:
                    reranked.append([])
                continue
            ids_np   = np.stack([ids for _, ids in items], axis=0)
            query_np = np.stack([query_qt[abs_i] for abs_i, _ in items], axis=0)
            ids_t    = torch.from_numpy(ids_np).to(device=dev)
            q_t      = torch.from_numpy(query_np).to(device=dev, dtype=torch.float32)
            c_t      = corpus_t[ids_t]
            mins     = torch.minimum(q_t[:, None, :], c_t).sum(dim=2)
            maxs     = q_t.sum(dim=1, keepdim=True) + corpus_sums_t[ids_t] - mins
            order    = torch.argsort(
                mins / maxs.clamp_min(1e-10), dim=1, descending=True
            ).cpu().numpy()
            for row, (_, ids) in zip(order, items):
                reranked.append(ids[row].tolist())
    return reranked


print("Eval functions defined.")

Eval functions defined.


In [7]:
# ── Run eval ─────────────────────────────────────────────────────────────────
if run_eval:
    if Path(ckpt_path).exists():
        model.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=True))
    model.eval()

    print("\n" + "=" * 72)
    print("MLP AUTOENCODER BASELINE — MSE reconstruction only, no WJ loss")
    print("=" * 72)

    embs        = generate_embeddings(model, qt, dev=device)
    corpus_embs = embs[:query_start]
    query_embs  = embs[query_start:]

    results = {}
    max_k   = max(max(candidate_ks), 500)

    t0 = time.time()
    nbrs = knn_intersection_gpu(
        query_embs, corpus_embs, k=max_k, dev=device,
        corpus_chunk=search_corpus_chunk
    )
    qps_stage1 = len(query_embs) / (time.time() - t0)
    ids_only   = [ids for ids, _ in nbrs]

    print(f"\n--- Stage 1: top-{max_k} by INTERSECTION on 512-D AE bottleneck ---")
    rec = eval_recall_dict(gt, ids_only, query_start, max_k)
    results["ae_no_rerank"] = {**rec, "qps": qps_stage1}
    for k, r in rec.items():
        print(f"  R@{k:<4} = {r:.4f}")
    print(f"  QPS ≈ {qps_stage1:.1f}")

    for k in candidate_ks:
        print(f"\n--- Stage 2: top-{k} AE candidates + raw WJ RATIO rerank ---")
        cand_ids = [ids[:k] for ids, _ in nbrs]
        t0 = time.time()
        rr_ids = rerank_wj_gpu(
            query_qt, cand_ids, corpus_qt, corpus_sums, device, rerank_batch_size
        )
        qps = len(query_embs) / (time.time() - t0)
        rec_rr = eval_recall_dict(gt, rr_ids, query_start, k)
        results[f"k{k}_raw_wj_ratio_rerank"] = {**rec_rr, "qps": qps, "k": k}
        for rk, rv in rec_rr.items():
            print(f"  R@{rk:<4} = {rv:.4f}")
        print(f"  QPS ≈ {qps:.1f}")

    payload = {
        dataset_name: results,
        "_meta": {
            "method": "MLP_autoencoder_mse_only",
            "bottleneck": 512,
            "loss": "MSE_reconstruction",
            "stage1_metric": "intersection_sum_min",
            "rerank": "raw_wj_ratio",
            "init": "scratch",
            "ckpt": ckpt_path,
            "time": __import__('time').strftime("%Y-%m-%d %H:%M:%S"),
        }
    }
    with open(out_path, "wb") as f:
        pickle.dump(payload, f)
    print(f"\nSaved {out_path}")

    print("\n--- Reference results ---")
    print("  nb 16 (intersection-min MLP): Stage-1 R@10=0.673 | K=500 R@10=0.996")
    print("  nb 18 (PCA):                  see /tmp/results_pca_baseline.pkl")

gc.collect()
if device.type == "cuda":
    torch.cuda.empty_cache()


MLP AUTOENCODER BASELINE — MSE reconstruction only, no WJ loss


Embedding: 100%|██████████| 20/20 [00:00<00:00, 114.98it/s]

simplex sums: min=1.0000 max=1.0000



KNN intersection: 100%|██████████| 32/32 [00:00<00:00, 223.74it/s]



--- Stage 1: top-1000 by INTERSECTION on 512-D AE bottleneck ---
  R@10   = 0.1525
  R@50   = 0.2360
  R@100  = 0.3018
  R@500  = 0.6288
  QPS ≈ 7866.3

--- Stage 2: top-500 AE candidates + raw WJ RATIO rerank ---


Raw WJ ratio rerank: 100%|██████████| 125/125 [00:00<00:00, 266.17it/s]


  R@10   = 0.8737
  R@50   = 0.7800
  R@100  = 0.7321
  R@500  = 0.6288
  QPS ≈ 3837.4

--- Stage 2: top-1000 AE candidates + raw WJ RATIO rerank ---


Raw WJ ratio rerank: 100%|██████████| 125/125 [00:00<00:00, 154.73it/s]


  R@10   = 0.9802
  R@50   = 0.9586
  R@100  = 0.9400
  R@500  = 0.8758
  QPS ≈ 2298.1

Saved /tmp/results_autoencoder_baseline.pkl

--- Reference results ---
  nb 16 (intersection-min MLP): Stage-1 R@10=0.673 | K=500 R@10=0.996
  nb 18 (PCA):                  see /tmp/results_pca_baseline.pkl
